In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Initialize LLM

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.7
)

## Initial Processing - Process the resume and user question to generate a single search query

In [ ]:
# ----- Resume Parsing ----- #
# ----- ----- ----- ----- #
from langchain_community.document_loaders import PyPDFLoader

resume = []
file_path = "C:/Users/adity/Certification Learning/LangChain/JobAI/Aditya_Resume_v1.pdf"
loader = PyPDFLoader(file_path)
for page in loader.lazy_load():
    resume.append(page)

In [ ]:
# ----- User Question ----- #
# ----- ----- ----- ----- #
user_input_1 = "Give jobs in the United states that are matching to my AstraZeneca experience"
user_input_2 = "Give me jobs in India that require AWS, Python and SQL based on my Deloitte experience"
user_input_3 = "Give me jobs in Texas, California and New York that require AWS, Python, SQL, Snowflake and AI/ML"
user_input_4 = "Jobs in JPMC in the field of Data Science matching Deloitte experience"

user_input = user_input_2

In [ ]:
# ----- Perform Initial Processing ----- #
# ----- ----- ----- ----- #
from langchain_core.prompts import ChatPromptTemplate

query_prompt = ChatPromptTemplate.from_template(
    """
    You are an expert AI recruiter. Analyze the user's resume and their specific question.
    
    TASK:
    Generate a structured search payload for a vector database.
    1. Identify the specific company/role mentioned in the User Question.
    2. If resume is available, extract 3-5 high-impact technical keywords or domain areas specifically associated with that experience.
    3. Generate a natural language search string combining the extracted resume themes and user-requested skills.
    
    CRITICAL: YOUR OUTPUT MUST FOLLOW THIS EXACT FORMAT. NO PREAMBLE, NO EXPLANATIONS.
    
    <QUERY>A single natural language search string for vector similarity (e.g. Senior Consultant with MDM and Data Governance experience proficient in AWS and Python)</QUERY>
    <COMPANY>The name of the organization mentioned (e.g. JPMC, Netflix), or "None" if not found.</COMPANY>
    <TITLE>The specific job title mentioned (e.g. Data Scientist, Manager), or "None" if not found.</TITLE>
    <LOCATION>The geographic location mentioned (e.g. United States, India), or "None" if not found.</LOCATION>
    <SKILL>A comma-separated list of technical skills mentioned in the question (e.g. AWS, Python, SQL), or "None" if not found.</SKILL>
    
    RESUME CONTENT:
    {resume}

    USER QUESTION:
    {question}

    OUTPUT:
    """
)

query_chain = query_prompt | llm
search_query = query_chain.invoke({"resume": resume, "question": user_input}).content
print(search_query)

In [ ]:
import re

def parse_search_payload(llm_output):
    tags = ["QUERY", "COMPANY", "TITLE", "LOCATION", "SKILL"]
    extracted = {}
    for tag in tags:
        match = re.search(f"<{tag}>(.*?)</{tag}>", llm_output, re.DOTALL)
        val = match.group(1).strip() if match else "None"
        extracted[tag] = None if val == "None" else val
    return extracted

clean_search_query = parse_search_payload(search_query)
print(clean_search_query)

## RAG Search

In [ ]:
def build_filter_dict(data):  
    # Mapping of LLM tags to your index metadata columns
    field_mapping = {
        'COMPANY': 'company',
        'TITLE': 'title',
        # 'LOCATION': 'locations',
        # 'SKILL': 'job_details'
    }

    filters = {}

    for tag, col_name in field_mapping.items():
        val = data.get(tag)

        # Skip empty or None values
        if not val or val == "None":
            continue

        # Convert comma-separated string → list
        if isinstance(val, str):
            items = [i.strip() for i in val.split(',') if i.strip()]
        else:
            items = [str(v).strip() for v in val if str(v).strip()]

        if not items:
            continue

        # For Standard endpoints:
        #   key = "column LIKE"
        #   value = single string OR list of strings
        key = f"{col_name} LIKE"

        # If only one value → pass string
        # If multiple → pass list (Databricks treats this as OR)
        filters[key] = items[0] if len(items) == 1 else items

    return filters


filter_str = build_filter_dict(clean_search_query)
print(filter_str)

In [ ]:
# ----- RAG ----- #
# ----- ----- ----- ----- #
from databricks.vector_search.client import VectorSearchClient
from databricks.vector_search.reranker import DatabricksReranker

client = VectorSearchClient(
    workspace_url="https://dbc-b544bb57-ffce.cloud.databricks.com/",
    # personal_access_token = userdata.get("DATABRICKS_TOKEN")
    disable_notice=True
  )

index = client.get_index(index_name="workspace.jobs.job_index")


results = index.similarity_search(
    query_text=clean_search_query["QUERY"],
    columns=["company", "title", "locations", "job_details", "url"],
    filters=filter_str,
    num_results=10,
    query_type = "hybrid",
    disable_notice=True 
)

In [ ]:
results

In [ ]:
# Create a context string from the search results
job_context = ""
for each_job in results['result']['data_array']:
    job_context += f"Job Info: {each_job[0]}\nLink: {each_job[1]}\n---\n"

## Final Responss - Process the user question and context to generate a final response 

In [ ]:
# ----- Final Response ----- #
# ----- ----- ----- ----- #
from langchain_core.prompts import ChatPromptTemplate

# The final prompt for the user
synthesis_prompt = ChatPromptTemplate.from_template(
    """
    You are a professional Career Assistant. Based ONLY on the provided job listings, 
    answer the user's question. If no relevant jobs are found, politely state that.
    
    Format your response with:
    1. A brief summary of why these jobs match their profile.
    2. A bulleted list of the top matches including the Job Title and the Link.
    3. Encouragement or a tip for applying.

    CONTEXT FROM DATABASE:
    {context}

    USER QUESTION:
    {question}

    YOUR RESPONSE:
    """
)

# Chain it together
synthesis_chain = synthesis_prompt | llm

# Get the final answer
final_response = synthesis_chain.invoke({
    "context": job_context, 
    "question": user_input
}).content

print(final_response)

In [ ]:
results

In [ ]:
headers = [col['name'] for col in results['manifest']['columns']]
combined_results = [
    dict(zip(headers, row)) 
    for row in data['result']['data_array']
]

In [ ]:
headers